# fastsae `hb_area()` engines on an NVIDIA GPU (Google Colab)

This notebook measures `hb_area()` with **INLA**, **JAX on the CPU**, **JAX on the CUDA GPU** and
**tipsae** (Stan, Beta model) on spatial (Besag ICAR) area-level models. The results feed the
pkgdown article
[hb_area() engines](https://ridsonap.github.io/fastsae/articles/hb-area-engines.html).

**Before running:** *Runtime → Change runtime type →* **R** and **T4 GPU**. Then *Runtime → Run all*.

Installation takes about 15–25 minutes. The benchmark then runs in the background for a few hours.
Keep this tab open. The results file is updated after every engine. If the runtime restarts, run
all cells again: finished parts are skipped.

In [ ]:
# 1. Check the GPU
cat(system("nvidia-smi", intern = TRUE), sep = "\n")

In [ ]:
# 2. System libraries for sf / spdep (needed by tipsae)
invisible(system("apt-get -qq update && apt-get -qq install -y libudunits2-dev libgdal-dev libgeos-dev libproj-dev python3-venv", intern = TRUE))
cat("done\n")

In [ ]:
# 3. R packages: binaries from Posit Package Manager, INLA from its own repository, fastsae from GitHub
codename <- system("lsb_release -cs", intern = TRUE)
options(
  repos = c(P3M = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
                          paste(getRversion(), R.version$platform, R.version$arch, R.version$os)),
  Ncpus = parallel::detectCores()
)
install.packages(c("remotes", "sf", "spdep", "tipsae", "reticulate", "ggplot2"))
install.packages("INLA", repos = c(getOption("repos"), INLA = "https://inla.r-inla-download.org/R/stable"),
                 dependencies = TRUE)
remotes::install_github("ridsonap/fastsae", upgrade = "never")
packageVersion("fastsae")

In [ ]:
# 4. Python side: JAX with CUDA 12 and NumPyro, in the virtualenv "r-numpyro-gpu"
Sys.unsetenv("RETICULATE_PYTHON")  # Colab may point reticulate at its own Python
fastsae::setup_numpyro_env(device = "cuda")
# Check in a fresh R process (JAX fixes its device per process): it must list a CUDA device
cat(system("Rscript -e 'fastsae::check_numpyro_available(device = \"cuda\")' 2>&1", intern = TRUE), sep = "\n")

In [ ]:
# 5. Start the benchmark in the background (INLA, JAX CPU, JAX CUDA, tipsae)
download.file("https://raw.githubusercontent.com/ridsonap/fastsae/main/benchmarks/bench_hb_area_engines.R",
              "bench_hb_area_engines.R", quiet = TRUE)
out <- "/content/hb_engine_benchmark_colab_t4.rds"
# The benchmark processes inherit this: use the JAX/NumPyro virtualenv from step 4
Sys.setenv(RETICULATE_PYTHON = reticulate::virtualenv_python("r-numpyro-gpu"))
system2("Rscript", c("bench_hb_area_engines.R", "--engines=inla,cpu,cuda,tipsae", paste0("--out=", out)),
        stdout = "bench.log", stderr = "bench.log", wait = FALSE)

In [ ]:
# 6. Progress: run this cell again to refresh. "exit" or "Saved" means finished.
log <- readLines("bench.log", warn = FALSE)
cat(grep(" n = |Error|Saved|failed", log, value = TRUE), sep = "\n")
cat("\nRunning:", any(grepl("bench_hb_area_engines", system("ps aux", intern = TRUE))), "\n")

In [ ]:
# 7. Results so far
res <- readRDS(out)
res$timings
res$agreement

**8. Download the results:** open the *Files* panel (folder icon on the left), right-click
`hb_engine_benchmark_colab_t4.rds` → *Download*. Send this file to be added to the article (it is
saved as `inst/extdata/hb_engine_benchmark_colab_t4.rds` in the repository).

If the session is about to time out, download the file anyway. It holds every finished engine,
and a later run with the file in `/content` continues from there.